# Định Giá Đúng (The Price is Right)

## Lộ trình Tuần 8

Ngày 1: Modal.com và SpecialistAgent  
Ngày 2: RAG, FrontierAgent, Ensemble Agent  
Ngày 3: ScannerAgent, MessengerAgent   
Ngày 4: AutonomousPlannerAgent  
Ngày 5: Chung kết The Price Is Right


Bây giờ đến lượt Planning Agent (Agent lập kế hoạch)

## 📝 Ghi chú tổng quan notebook

### Tóm tắt quy trình của notebook
Notebook này xây dựng `AutonomousPlanningAgent` - agent có khả năng tự lập kế hoạch bằng cách gọi các "tool" (function calling). Quy trình: (1) thử nghiệm với dữ liệu deal mẫu (test data), (2) định nghĩa 3 hàm giả lập (scan/estimate/notify) để hiểu cơ chế tool trước, (3) khai báo các function này dưới dạng JSON schema để LLM có thể gọi, (4) xây dựng vòng lặp cho LLM tự quyết định gọi tool nào, (5) thay các hàm giả bằng hàm thật và đóng gói thành `AutonomousPlanningAgent`.

### Ý nghĩa chính của notebook
Notebook giải quyết bài toán: làm sao để một LLM có thể tự động điều phối nhiều bước (quét deal → ước tính giá trị thật → thông báo cho người dùng) mà không cần con người viết logic điều khiển cứng. Đây chính là kỹ thuật "function calling" / "tool use" - nền tảng của các AI Agent hiện đại.

### Mục tiêu cuối cùng
Sau khi chạy xong notebook, ta có được `AutonomousPlanningAgent` - agent có thể tự gọi `ScannerAgent`, ước tính giá trị thật của sản phẩm (dùng Chroma + RAG từ Ngày 2), và tự quyết định khi nào cần thông báo cho người dùng về deal tốt nhất, sẵn sàng để tích hợp vào `DealAgentFramework` hoàn chỉnh.

In [ ]:
# Import thư viện cần thiết: json để xử lý dữ liệu tool call, OpenAI để gọi LLM,
# ScannerAgent từ Ngày 3, chromadb để truy cập vector store từ Ngày 2, logging để ghi log.
# Nạp biến môi trường, khởi tạo client OpenAI và chọn model gpt-5.1.

import json
from openai import OpenAI
from dotenv import load_dotenv
from agents.scanner_agent import ScannerAgent
import chromadb
import logging
load_dotenv(override=True)
openai = OpenAI()
MODEL = "gpt-5.1"

## Bắt đầu với một số dữ liệu mẫu (test data)

In [ ]:
# Lấy dữ liệu deal mẫu (test) từ ScannerAgent để thử nghiệm logic của Planning Agent
# mà không cần chờ quét dữ liệu thật mỗi lần chạy thử.
test_results = ScannerAgent().test_scan()
test_results

## Bây giờ hãy tạo 3 hàm giả lập (pretend functions)..

In [ ]:
# Hàm giả lập (fake/pretend) đầu tiên: thay vì quét internet thật, hàm này chỉ trả về
# dữ liệu test_results đã có sẵn - giúp thử nghiệm cơ chế function calling trước khi
# thay bằng hàm thật ở cuối notebook.
def scan_the_internet_for_bargains() -> str:
    """ This tool scans the internet for great deals and gets a curated list of promising deals """
    print("Fake function to scan the internet - this returns a hardcoded set of deals")
    return test_results.model_dump_json()

In [ ]:
# Hàm giả lập thứ hai: thay vì thật sự ước tính giá trị sản phẩm (bằng RAG/mô hình),
# hàm này luôn trả về $300 cố định - chỉ để kiểm tra luồng gọi tool của LLM.
def estimate_true_value(description: str) -> str:
    """
    This tool estimates the true value of a product based on a text description of it
    """
    print(f"Fake function to estimating true value of {description[:20]}... - this always returns $300")
    return f"Product {description} has an estimated true value of $300"

In [ ]:
# Hàm giả lập thứ ba: thay vì gửi thông báo thật (qua Pushover), hàm này chỉ in ra
# console để xác nhận rằng LLM đã gọi đúng tool với đúng tham số.
def notify_user_of_deal(description: str, deal_price: float, estimated_true_value: float, url: str) -> str:
    """
    This tool notifies the user of a great deal, given a description of it, the price of the deal, and the estimated true value
    """
    print(f"Fake function to notify user of {description} which costs {deal_price} and estimate is {estimated_true_value}")
    return "notification sent ok"

### Hãy thử chạy các hàm này

In [ ]:
# Gọi thử trực tiếp hàm notify_user_of_deal() (chưa qua LLM) để kiểm tra hàm chạy đúng.
notify_user_of_deal("a new iphone", 100, 1000, "https://www.apple.com/iphone")

### Được rồi, giờ đến một khối JSON lớn

In [ ]:
# Khai báo "schema" JSON cho 3 tool ở trên theo đúng định dạng OpenAI function calling
# yêu cầu: mỗi tool có "name", "description" (LLM dùng để biết khi nào nên gọi tool này)
# và "parameters" (mô tả các tham số đầu vào). Các trường description giữ nguyên tiếng Anh
# vì đây là nội dung được gửi trực tiếp cho LLM.

scan_function = {
        "name": "scan_the_internet_for_bargains",
        "description": "Returns top bargains scraped from the internet along with the price each item is being offered for",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False
        }
    }

estimate_function = {
    "name": "estimate_true_value",
    "description": "Given the description of an item, estimate how much it is actually worth",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {
                "type": "string",
                "description": "The description of the item to be estimated"
            },
        },
        "required": ["description"],
        "additionalProperties": False
    }
}

notify_function = {
    "name": "notify_user_of_deal",
    "description": "Send the user a push notification about the single most compelling deal; only call this one time",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {
                "type": "string",
                "description": "The description of the item itself scraped from the internet"
            },
            "deal_price": {
                "type": "number",
                "description": "The price offered by this deal scraped from the internet"
            }
            ,
            "estimated_true_value": {
                "type": "number",
                "description": "The estimated actual value that this is worth"
            }
            ,
            "url": {
                "type": "string",
                "description": "The URL of this deal as scraped from the internet"
            }
        },
        "required": ["description", "deal_price", "estimated_true_value", "url"],
        "additionalProperties": False
    }
}

In [ ]:
# Gom cả 3 function schema ở trên thành danh sách "tools" theo đúng định dạng mà
# OpenAI Chat Completions API yêu cầu khi dùng tính năng function calling.
tools = [{"type": "function", "function": scan_function},
 {"type": "function", "function": estimate_function},
 {"type": "function", "function": notify_function}
 ]

In [ ]:
# In ra để kiểm tra lại cấu trúc của danh sách tools.
tools

In [ ]:
# Hàm này xử lý các tool_call mà LLM yêu cầu: với mỗi lệnh gọi tool, lấy tên hàm,
# parse tham số từ JSON, tìm hàm tương ứng trong globals() rồi thực thi thật sự,
# sau đó đóng gói kết quả theo định dạng "role: tool" để gửi lại cho LLM ở vòng tiếp theo.
def handle_tool_call(message):
    """
    Actually call the tools associated with this message
    """
    results = []
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
# Định nghĩa system_message và user_message (giữ nguyên tiếng Anh vì là prompt gửi cho LLM)
# hướng dẫn LLM: quét deal → ước tính giá trị thật → chọn deal tốt nhất → thông báo.
# Đây chính là "kế hoạch" mà agent cần tự thực hiện bằng cách gọi các tool ở trên.

system_message = "You find great deals on bargain products using your tools, and notify the user of the best bargain."
user_message = """
First, use your tool to scan the internet for bargain deals. Then for each deal, use your tool to estimate its true value.
Then pick the single most compelling deal where the price is much lower than the estimated true value, and use your tool to notify the user.
Then just reply OK to indicate success.
"""
messages = [{"role": "system", "content": system_message},{"role": "user", "content": user_message}]

In [ ]:
# Kiểm tra lại nội dung messages trước khi gửi cho LLM.
messages

In [ ]:
# Đây là vòng lặp "agentic": lặp đi lặp lại việc gọi LLM cho đến khi LLM không còn
# yêu cầu gọi thêm tool nào nữa (finish_reason khác "tool_calls"). Mỗi khi LLM yêu cầu
# gọi tool, ta thực thi tool thật (handle_tool_call) rồi gửi kết quả ngược lại cho LLM
# để nó tiếp tục suy luận bước kế tiếp - đây chính là cơ chế cốt lõi của một AI Agent.
done = False
while not done:
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        results = handle_tool_call(message)
        messages.append(message)
        messages.extend(results)
    else:
        done = True
response.choices[0].message.content

## Và bây giờ.. chuyển sang một Autonomous Planning Agent thực thụ

Và thay các hàm giả lập bằng các hàm THẬT!

In [ ]:
# Bật ghi log ở mức INFO để theo dõi hoạt động bên trong AutonomousPlanningAgent.
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
# Kết nối tới Chroma vector store đã được tạo ở Ngày 2 (thư mục products_vectorstore),
# lấy lại collection "products" để dùng cho việc ước tính giá trị thật (RAG).
DB = "products_vectorstore"
client = chromadb.PersistentClient(path=DB)
collection = client.get_or_create_collection('products')

In [ ]:
# Import và khởi tạo AutonomousPlanningAgent thật, truyền vào collection Chroma
# để agent có thể tự dùng RAG (từ Ngày 2) khi ước tính giá trị thật của sản phẩm.
from agents.autonomous_planning_agent import AutonomousPlanningAgent
agent = AutonomousPlanningAgent(collection)

In [ ]:
# Chạy toàn bộ quy trình tự động: agent sẽ tự quét deal, ước tính giá trị thật,
# và thông báo cho người dùng nếu tìm thấy deal đủ hấp dẫn - không cần can thiệp thủ công.
agent.plan()